In [1]:
from pathlib import Path

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

Repo root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/4. Semester/DEDA Project/DEDA_LLM_Spatial_Hotelling


In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
import sys
import json
from shapely.geometry import shape
from hotelling.spatial.admin import join_lor_names

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from hotelling.spatial.boundaries import load_boundary

PATH_RAW = REPO_ROOT / Path('data/raw')
PATH_PROCESSED = REPO_ROOT / Path('data/processed')

# Midpoint table (center coordinates)
zensus = gpd.read_parquet(PATH_RAW / 'zensus2022_grid.parquet')
zensus_filtered = gpd.read_parquet(PATH_RAW / 'zensus2022_grid_filtered.parquet')
lor = gpd.read_parquet(PATH_PROCESSED / 'lor.parquet')

# CRITICAL FIX: berlin.geojson has EPSG:3035 coordinates but geopandas auto-detects as EPSG:4326
# We must force the correct CRS instead of transforming from the wrong one
with open(PATH_RAW / 'city_boundary_Berlin.geojson', 'r') as f:
    berlin_json = json.load(f)
berlin = gpd.GeoDataFrame([1], geometry=[shape(berlin_json['geometry'])], crs='EPSG:3035')

boundary = load_boundary(PATH_RAW / 'relation_boundary_14983.geojson')

In [3]:
# Load pop_grid

grid = gpd.read_parquet(PATH_PROCESSED / 'pop_grid.parquet')

# Build squares from points of grid
grid['geometry'] = grid.apply(lambda row: row.geometry.buffer(50, cap_style='square'), axis=1)
grid['index'] = grid.index

In [4]:
IHK = pd.read_csv(PATH_RAW / '2023_12_IHK_Berlin_Gewerbedaten.csv')

# To geodataframe based on latitude and longitude columns
IHK_gdf = gpd.GeoDataFrame(
    IHK,
    geometry=gpd.points_from_xy(IHK['longitude'], IHK['latitude']),
    crs='EPSG:4326'
).to_crs('EPSG:3035')

In [5]:
# from e.g. '1 - 3 Beschäftigte' to mean(1,3) = 2
def parse_employees_range(range_str):
    if pd.isna(range_str):
        return np.nan
    
    if range_str == 'unbekannt':
        return np.nan
    
    try:
        parts = range_str.split('-')
        if len(parts) == 2:
            low = int(parts[0].strip())
            high = int(parts[1].strip().split()[0])  # Remove 'Beschäftigte'
            return (low + high) / 2
        else:
            return int(range_str.strip().split()[0])  # Single value case
    except Exception as e:
        print(f"Error parsing '{range_str}': {e}")
        return np.nan

IHK_gdf['empl'] = IHK_gdf['employees_range'].apply(parse_employees_range)

In [6]:
# Assign to grid cells - preserving grid cell ID in IHK_gdf for filtering
IHK_gdf = gpd.sjoin(IHK_gdf, grid, how='left', predicate='within')

# Aggregate employment by grid cell
empl_by_cell = IHK_gdf.groupby('index')['empl'].sum().reset_index()
empl_by_cell.columns = ['index', 'empl']

# Merge employment data back to grid
grid = grid.reset_index(drop=True).merge(empl_by_cell, left_index=True, right_on='index', how='left')
grid['empl'] = grid['empl'].fillna(0)

In [ ]:
# Load the gebaeude and stadtstruktur data
gebaeude = gpd.read_file(PATH_RAW / 'gebaeude.gpkg')
stadtstruktur = gpd.read_file(PATH_RAW / 'stadtstruktur.gpkg')
zentren_fma = gpd.read_file(PATH_RAW / 'zentren.gpkg', layer="zentren_fma")
zentren_zh = gpd.read_file(PATH_RAW / 'zentren.gpkg', layer="zentren_zh")

/Users/jedrek/miniforge3/envs/py314/lib/python3.14/site-packages/pyogrio/geopandas.py:382: UserWarning: More than one layer found in 'zentren.gpkg': 'zentren_fma' (default), 'zentren_zh'. Specify layer parameter to avoid this warning.
  result = read_func(
